# Conditional Probability & Bayes' Theorem

Two connected ideas:
- **Conditional probability** — how new information changes the odds
- **Bayes' theorem** — how to *flip* a conditional around

These are the heart of probability, and the foundation of Bayesian methods later.

## Part 1: Conditional Probability

**The idea:** *"Now that I know B happened, what's the chance of A?"*

New information **changes the odds**. Written **P(A | B)** — "probability of A **given** B."

**Everyday example:** What's the chance a random person is a pro basketball player? Tiny.
Now I tell you they're **over 7 feet tall** — suddenly that chance jumps way up.
Same person, but the extra info changed the probability.

### The intuition: zoom into a smaller world 🔍
Once you know B happened, **B's world becomes your entire universe.** You throw away
everything else and ask: *within this smaller world, how often is A also true?*

**A room of 100 people:**
- 40 wear hats
- 20 wear glasses
- 10 wear **both**

**P(glasses | hat)?** Zoom into the 40 hat-wearers. Of those, 10 wear glasses → **10/40 = 0.25**
You ignored the other 60 people entirely. That's the zoom.

### The formula
$$P(A \mid B) = \frac{P(A \text{ and } B)}{P(B)}$$
"How often A and B happen together" ÷ "how often B happens at all" — i.e. the overlap,
divided by the world you zoomed into.

In [2]:
import numpy as np

# Build the room: 100 people, 40 hats, 20 glasses, 10 both
N = 100
hat     = np.zeros(N, dtype=bool)
glasses = np.zeros(N, dtype=bool)
hat[:40] = True              # people 0-39 wear hats
glasses[30:50] = True        # people 30-49 wear glasses -> overlap is 30-39 (10 people)

print("total people       :", N)
print("wear hats          :", hat.sum())
print("wear glasses       :", glasses.sum())
print("wear BOTH (overlap):", (hat & glasses).sum())

# Conditional probability = zoom into one world, count the overlap there
p_glasses_given_hat = (hat & glasses).sum() / hat.sum()
p_hat_given_glasses = (hat & glasses).sum() / glasses.sum()

print("\nZoom into the HAT world (40 people) -> how many wear glasses?")
print("  P(glasses | hat) = 10/40 =", p_glasses_given_hat)
print("Zoom into the GLASSES world (20 people) -> how many wear hats?")
print("  P(hat | glasses) = 10/20 =", p_hat_given_glasses)

total people       : 100
wear hats          : 40
wear glasses       : 20
wear BOTH (overlap): 10

Zoom into the HAT world (40 people) -> how many wear glasses?
  P(glasses | hat) = 10/40 = 0.25
Zoom into the GLASSES world (20 people) -> how many wear hats?
  P(hat | glasses) = 10/20 = 0.5


## Bayes' Theorem — flipping the conditional

Look carefully at what just happened above:

| | numerator (the overlap) | denominator (the world) | answer |
|---|---|---|---|
| P(glasses \| hat) | **10** | 40 | 0.25 |
| P(hat \| glasses) | **10** | 20 | 0.50 |

**The overlap is the SAME (10 people).** Only the **world we divided by** changed.

👉 **That's Bayes.** Two conditionals are just *the same overlap, divided by two different
worlds.* Flipping a conditional = keep the overlap, swap the denominator.

### The formula
$$P(A \mid B) = \frac{P(B \mid A)\times P(A)}{P(B)}$$

Read with "zoom" eyes:
- **P(B|A) × P(A)** → rebuild the overlap using the *other* world
- **÷ P(B)** → now zoom into *this* world instead

### Why bother flipping?
Because **the direction you can measure is often not the direction you want.**
- Easy to measure: P(test positive | you're sick) — labs test known-sick people and count.
- What you actually want: P(you're sick | test positive) — you just got a positive result!

Bayes converts the one you have into the one you need.

In [3]:
# Bayes: P(A|B) = P(B|A) * P(A) / P(B)
p_hat     = hat.sum() / N
p_glasses = glasses.sum() / N

bayes_result = (p_glasses_given_hat * p_hat) / p_glasses

print("Flip P(glasses|hat) into P(hat|glasses) using Bayes:")
print(f"  = {p_glasses_given_hat} * {p_hat} / {p_glasses} = {bayes_result}")
print(f"Direct count gave: {p_hat_given_glasses}  ->  MATCH ✓")

Flip P(glasses|hat) into P(hat|glasses) using Bayes:
  = 0.25 * 0.4 / 0.2 = 0.5
Direct count gave: 0.5  ->  MATCH ✓


## The famous surprise — why the *base rate* matters

A disease affects **1 in 1000** people. A test is **99% accurate**.
You test **positive**. What's the chance you're actually sick?

Most people say "99%!" — **it's about 9%.** Here's why:

Out of 100,000 people:
- **100** are actually sick → ~99 test positive (true positives)
- **99,900** are healthy → but 1% get a false positive → ~999 test positive

So ~1,098 people test positive, and only 99 are truly sick → **99/1098 ≈ 9%**

**The reason:** the healthy group is *enormous*, so even a small error rate produces far
more false positives than there are true positives. The **base rate** (how rare the disease
is) dominates the answer — and Bayes forces you to account for it. Intuition forgets it.

In [4]:
population = 100_000
n_sick    = population // 1000        # 1 in 1000
n_healthy = population - n_sick

true_positives  = int(n_sick * 0.99)      # 99% of sick test positive
false_positives = int(n_healthy * 0.01)   # 1% of healthy ALSO test positive

print(f"actually sick : {n_sick}")
print(f"healthy       : {n_healthy}")
print(f"\ntrue positives  (sick, test+)   : {true_positives}")
print(f"false positives (healthy, test+): {false_positives}   <- the huge healthy group!")

total_positive = true_positives + false_positives
p_sick_given_pos = true_positives / total_positive
print(f"\nP(sick | positive) = {true_positives}/{total_positive} = {p_sick_given_pos:.4f}")
print(f"  -> about {p_sick_given_pos*100:.1f}%, NOT 99%!")

# same answer straight from the Bayes formula
p_disease = 1/1000
p_pos = 0.99*p_disease + 0.01*(1-p_disease)
print(f"\nVia Bayes formula: (0.99 * 0.001) / {p_pos:.5f} = {(0.99*p_disease)/p_pos:.4f}  -> MATCH ✓")

actually sick : 100
healthy       : 99900

true positives  (sick, test+)   : 99
false positives (healthy, test+): 999   <- the huge healthy group!

P(sick | positive) = 99/1098 = 0.0902
  -> about 9.0%, NOT 99%!

Via Bayes formula: (0.99 * 0.001) / 0.01098 = 0.0902  -> MATCH ✓


## Takeaway

**Conditional probability** = zoom into the world where B is true, then count how often A
happens *there*. New info shrinks your universe and changes the odds.

**Bayes' theorem** = both directions share the **same overlap** — only the world you divide
by changes. So you can flip a conditional by rebuilding the overlap and re-dividing:
$$P(A\mid B) = \frac{P(B\mid A)\,P(A)}{P(B)}$$

**Why it matters:** the direction you can *measure* usually isn't the direction you *want*.
And Bayes forces you to include the **base rate** — which is why a positive test for a rare
disease is far less alarming than it feels.